# EstateInsight (Pumpkin Grill) - Data Pipeline
สมุดโน้ตนี้ถูกตั้งค่าให้รันทีละขั้นตอน (Step-by-step) เพื่อโชว์ผลลัพธ์และดูหน้าตาข้อมูลในแต่ละขั้นให้เห็นภาพชัดเจนครับ

In [ ]:
import os
import pandas as pd
from IPython.display import display

# สร้างโฟลเดอร์ที่จำเป็น
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/cleaned', exist_ok=True)
os.makedirs('data/transformed', exist_ok=True)
os.makedirs('models', exist_ok=True)



## 1. Data Extraction (จำลองข้อมูลดิบ)

In [ ]:
import os
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

def generate_mock_data(base_path: str, seed: int = 42):
    """Generates mock real estate data with injected problems."""
    np.random.seed(seed)
    random.seed(seed)
    
    raw_dir = os.path.join(base_path, 'data', 'raw')
    os.makedirs(raw_dir, exist_ok=True)
    
    # Common lists
    locations = [
        'สุขุมวิท', 'Sukhumvit', 'สีลม', 'Silom', 'สาทร', 'Sathorn', 'ลาดพร้าว', 'Lat Phrao',
        'พระราม9', 'Rama 9', 'อ่อนนุช', 'On Nut', 'บางนา', 'Bang Na', 'รังสิต', 'Rangsit',
        'ธนบุรี', 'Thonburi', 'พญาไท', 'Phaya Thai', 'อารีย์', 'Ari', 'ทองหล่อ', 'Thong Lo',
        'เอกมัย', 'Ekkamai', 'ราชเทวี', 'Ratchathewi', 'ห้วยขวาง', 'Huai Khwang',
        'บางกะปิ', 'Bang Kapi', 'มีนบุรี', 'Min Buri', 'หนองจอก', 'Nong Chok',
        'ตลิ่งชัน', 'Taling Chan', 'บางแค', 'Bang Khae', 'จตุจักร', 'Chatuchak'
    ]
    prop_types = ['คอนโด', 'Condo', 'condominium', 'บ้านเดี่ยว', 'House', 'Single House', 'detached house', 'ทาวน์โฮม', 'Townhome', 'Townhouse', 'ที่ดิน', 'Land', 'land plot']
    
    def random_date(start_days_ago=365):
        base_date = datetime.now() - timedelta(days=random.randint(0, start_days_ago))
        return base_date
        
    # Zone multiplier for price-per-sqm (CBD is most expensive)
    zone_price_multiplier = {
        'สุขุมวิท': 1.8, 'Sukhumvit': 1.8, 'สีลม': 1.7, 'Silom': 1.7,
        'สาทร': 1.65, 'Sathorn': 1.65, 'ทองหล่อ': 2.0, 'Thong Lo': 2.0,
        'เอกมัย': 1.6, 'Ekkamai': 1.6, 'อารีย์': 1.5, 'Ari': 1.5,
        'พญาไท': 1.3, 'Phaya Thai': 1.3, 'ราชเทวี': 1.25, 'Ratchathewi': 1.25,
        'ลาดพร้าว': 1.1, 'Lat Phrao': 1.1, 'พระราม9': 1.2, 'Rama 9': 1.2,
        'อ่อนนุช': 1.05, 'On Nut': 1.05, 'ห้วยขวาง': 1.1, 'Huai Khwang': 1.1,
        'จตุจักร': 1.15, 'Chatuchak': 1.15, 'บางนา': 0.85, 'Bang Na': 0.85,
        'บางกะปิ': 0.8, 'Bang Kapi': 0.8, 'ธนบุรี': 0.75, 'Thonburi': 0.75,
        'ตลิ่งชัน': 0.7, 'Taling Chan': 0.7, 'บางแค': 0.65, 'Bang Khae': 0.65,
        'รังสิต': 0.55, 'Rangsit': 0.55, 'มีนบุรี': 0.5, 'Min Buri': 0.5,
        'หนองจอก': 0.4, 'Nong Chok': 0.4
    }
    
    # Base price-per-sqm by property type (THB)
    type_base_price = {
        'คอนโด': 120000, 'Condo': 120000, 'condominium': 120000,
        'บ้านเดี่ยว': 80000, 'House': 80000, 'Single House': 80000, 'detached house': 80000,
        'ทาวน์โฮม': 65000, 'Townhome': 65000, 'Townhouse': 65000,
        'ที่ดิน': 45000, 'Land': 45000, 'land plot': 45000
    }
    
    bts_stations = [
        'BTS สุขุมวิท', 'BTS สีลม', 'BTS อ่อนนุช', 'BTS เอกมัย', 'BTS ทองหล่อ',
        'BTS พร้อมพงษ์', 'BTS อารีย์', 'BTS สะพานควาย', 'BTS ชิดลม',
        'MRT ลาดพร้าว', 'MRT พระราม9', 'MRT ห้วยขวาง', 'MRT สุขุมวิท',
        'MRT จตุจักร', 'MRT บางซื่อ', 'MRT ศูนย์วัฒนธรรม'
    ]
    
    def generate_base_listing(source_name, id_prefix, p_type):
        loc = random.choice(locations)
        title = f"{p_type} for sale at {loc}"
        desc = f"Beautiful {p_type} located in {loc}. Great view."
        
        # Area based on property type
        if p_type in ['คอนโด', 'Condo', 'condominium']:
            area = random.uniform(25, 200)
            beds = max(1, int(area / 40) + random.randint(-1, 1))
            beds = min(beds, 5)
            baths = max(1, beds - random.randint(0, 1))
            floor = random.randint(1, 40)
        elif p_type in ['บ้านเดี่ยว', 'House', 'Single House', 'detached house']:
            area = random.uniform(100, 500)
            beds = max(2, int(area / 60) + random.randint(-1, 1))
            beds = min(beds, 8)
            baths = max(2, beds - random.randint(0, 2))
            floor = random.randint(1, 3)
        elif p_type in ['ทาวน์โฮม', 'Townhome', 'Townhouse']:
            area = random.uniform(60, 200)
            beds = max(2, int(area / 50) + random.randint(-1, 1))
            beds = min(beds, 5)
            baths = max(1, beds - random.randint(0, 1))
            floor = random.randint(2, 4)
        else:  # Land
            area = random.uniform(50, 5000)
            beds = 0
            baths = 0
            floor = 0
        
        # Price = area × base_price_per_sqm × zone_multiplier × noise
        base_ppsqm = type_base_price.get(p_type, 80000)
        zone_mult = zone_price_multiplier.get(loc, 1.0)
        noise = random.gauss(1.0, 0.15)  # ±15% random variation
        
        # BTS proximity bonus
        near_bts = random.choice(bts_stations) if random.random() < 0.6 else ''
        bts_bonus = 1.1 if near_bts else 1.0
        
        # Floor bonus for condos
        floor_bonus = 1.0 + (floor * 0.005) if p_type in ['คอนโด', 'Condo', 'condominium'] else 1.0
        
        price = area * base_ppsqm * zone_mult * noise * bts_bonus * floor_bonus
        price = max(price, 100000)  # Minimum price floor
        
        dt = random_date()
        
        return {
            'price': price,
            'area_sqm': area,
            'beds': beds,
            'baths': baths,
            'floor': floor,
            'loc': loc,
            'title': title,
            'desc': desc,
            'near_bts': near_bts,
            'dt': dt
        }

    print("Generating Source 1: DDProperty (17,000 records)...")
    dd_records = []
    for i in range(17000):
        p_type = random.choice(prop_types)
        base = generate_base_listing("DDProperty", "DD", p_type)
        
        price = int(base['price'])
        area = base['area_sqm']
        beds = base['beds']
        
        # Inject problems
        r = random.random()
        if r < 0.08:
            price = np.nan
        if r < 0.05:
            area = np.nan
        if 0.13 < r < 0.16:
            price = -abs(price) if price is not np.nan else price
            area = -abs(area) if area is not np.nan else area
        if 0.16 < r < 0.18:
            price = 0
        if 0.18 < r < 0.20 and price is not np.nan and area is not np.nan:
            # Swap
            price, area = area, price
        if 0.20 < r < 0.22:
            beds = random.randint(21, 50)
            
        title = base['title']
        desc = base['desc']
        if r < 0.05: title += ' &amp; <br>'
        if r < 0.10: title += ' 🏠🔥⭐'
        if 0.10 < r < 0.15: title = f"คอนโดสวย โทร 081-{random.randint(100,999)}-{random.randint(1000,9999)} " + title
            
        record = {
            'id': f'DD{i}',
            'title': title,
            'description': desc,
            'price': price,
            'area_sqm': area,
            'bedrooms': beds,
            'bathrooms': base['baths'],
            'property_type': p_type,
            'location': base['loc'],
            'floor': base['floor'],
            'near_bts': base['near_bts'],
            'posted_date': base['dt'].strftime('%Y-%m-%d'),
            'url': f'https://ddproperty.fake/listing/{i}',
            'source': 'DDProperty'
        }
        dd_records.append(record)
        
    # Duplicate some rows
    for _ in range(200):
        idx = random.randint(0, len(dd_records)-1)
        dd_records.append(dd_records[idx].copy())
        
    pd.DataFrame(dd_records).to_csv(os.path.join(raw_dir, 'ddproperty.csv'), index=False)

    print("Generating Source 2: Baania (17,000 records)...")
    baania_records = []
    # Mix thai formats
    for i in range(17000):
        p_type = random.choice(prop_types)
        base = generate_base_listing("Baania", "BN", p_type)
        
        price_val = base['price']
        price_str = f"{int(price_val)}"
        
        r = random.random()
        if r < 0.1: price_str = f"{price_val/1_000_000:.1f} ล้าน"
        elif r < 0.2: price_str = f"{int(price_val/100_000)} แสน"
        elif r < 0.3: price_str = f"{int(price_val):,} บาท"
        elif r < 0.35: price_str = "ราคาต่อรอง"
        elif r < 0.4: price_str = "สอบถาม"
        
        beds = base['beds']
        if r < 0.10: beds = np.nan
        
        area_wah = base['area_sqm'] / 4.0
        if r < 0.05: area_wah = 999999
        
        title = base['title']
        desc = base['desc']
        if r < 0.05: title = "ด่วน!!! " * 3 + title
        if 0.05 < r < 0.10: 
            desc = desc.upper() + " Call 088888888"
            desc = "รับฝากขาย-เช่า ติดต่อ... " + desc
            
        loc = base['loc']
        if loc == 'Sukhumvit' or loc == 'สุขุมวิท':
            loc = random.choice(['สุขุมวิท', 'สุขุมวิท ', 'Sukhumvit', ' สุขุมวิท', 'สุขุมวิทร์'])
            
        # Mixed dates
        dt_str = base['dt'].strftime('%Y-%m-%d')
        if r < 0.3: dt_str = base['dt'].strftime('%d/%m/%Y')
        elif r < 0.6: dt_str = base['dt'].strftime('%d %b %y')
            
        record = {
            'รหัส': f'BN{i}',
            'หัวข้อ': title,
            'รายละเอียด': desc,
            'ราคา': price_str,
            'พื้นที่_ตร.วา': area_wah,
            'ห้องนอน': beds,
            'ห้องน้ำ': base['baths'],
            'ประเภท': p_type,
            'ทำเล': loc,
            'ชั้น': base['floor'],
            'ใกล้รถไฟฟ้า': base['near_bts'],
            'วันที่ลง': dt_str,
            'ลิงก์': f'https://baania.fake/p/{i}',
            'แหล่ง': 'Baania'
        }
        baania_records.append(record)
        
    for _ in range(150):
        idx = random.randint(0, len(baania_records)-1)
        dup = baania_records[idx].copy()
        dup['ราคา'] = str(dup['ราคา']) + ' '
        baania_records.append(dup)

    pd.DataFrame(baania_records).to_csv(os.path.join(raw_dir, 'baania.csv'), index=False)
    
    print("Generating Source 3: HipFlat (16,000 records)...")
    hipflat_records = []
    for i in range(16000):
        p_type = random.choice(prop_types)
        base = generate_base_listing("HipFlat", "HF", p_type)
        
        cost_usd = base['price'] / 35.0
        
        r = random.random()
        if r < 0.05: cost_usd = base['price'] # accidentaly in THB
        if 0.05 < r < 0.08: cost_usd = random.choice([0, 1])
        
        sqm = base['area_sqm']
        land_rai = sqm // 1600
        land_ngan = (sqm % 1600) // 400
        land_wah = ((sqm % 1600) % 400) / 4.0
        
        if r < 0.07:
            land_rai, land_ngan, land_wah = np.nan, np.nan, np.nan
        elif 0.07 < r < 0.12:
            land_rai += 0.5 # Fractional rai double counting
            
        type_val = p_type
        if r < 0.03: type_val = random.choice(['asdf', '123', ''])
        
        beds_val = base['beds']
        if r < 0.05: beds_val = 'studio'
        elif 0.05 < r < 0.1: beds_val = '3+1'
        elif 0.1 < r < 0.15: beds_val = 'N/A'
        
        dt_str = base['dt'].strftime('%Y-%m-%d')
        if r < 0.3: dt_str = base['dt'].strftime('%b %d, %Y')
        elif r < 0.6: dt_str = base['dt'].strftime('%d/%m/%y')
        
        name = base['title']
        if r < 0.05: name += '\u0e00'
        
        record = {
            'listing_id': f'HF{i}',
            'name': name,
            'desc': base['desc'],
            'cost_usd': cost_usd,
            'land_rai': land_rai,
            'land_ngan': land_ngan,
            'land_wah': land_wah,
            'beds': beds_val,
            'baths': base['baths'],
            'type': type_val,
            'area': sqm,
            'level': base['floor'],
            'date': dt_str,
            'platform': 'HipFlat'
        }
        hipflat_records.append(record)

    pd.DataFrame(hipflat_records).to_csv(os.path.join(raw_dir, 'hipflat.csv'), index=False)
    print("Done generating mock data.")



# รันการจำลองข้อมูล
print("--- Step 1: Data Generation ---")
generate_mock_data('.', seed=42)

# ดูหน้าตาข้อมูลดิบ (เช่นของ DDProperty ที่ยังมีความพังอยู่)
print("\nตัวอย่างข้อมูลดิบ (DDProperty):")
df_raw = pd.read_csv('data/raw/ddproperty.csv')
display(df_raw.head())



## 2. Data Cleaning (ทำความสะอาดข้อมูล)

In [ ]:
import os
import pandas as pd
import numpy as np
import re

class DataCleaner:
    """Handles cleaning of real estate listings data."""
    
    def __init__(self, raw_dir, clean_dir):
        self.raw_dir = raw_dir
        self.clean_dir = clean_dir
        self.stats = {'before': 0, 'after': 0, 'operations': {}}
        
    def _log_op(self, op_name, count_diff):
        self.stats['operations'][op_name] = self.stats['operations'].get(op_name, 0) + count_diff
        print(f"  [{op_name}] Removed {count_diff} records")

    def _extract_location_from_title(self, title):
        """Extract location name from title text like 'Condo for sale at สุขุมวิท'."""
        if pd.isna(title):
            return np.nan
        locations = [
            'สุขุมวิท', 'Sukhumvit', 'สีลม', 'Silom', 'สาทร', 'Sathorn',
            'ลาดพร้าว', 'Lat Phrao', 'พระราม9', 'Rama 9', 'อ่อนนุช', 'On Nut',
            'บางนา', 'Bang Na', 'รังสิต', 'Rangsit', 'ธนบุรี', 'Thonburi',
            'พญาไท', 'Phaya Thai', 'อารีย์', 'Ari', 'ทองหล่อ', 'Thong Lo',
            'เอกมัย', 'Ekkamai', 'ราชเทวี', 'Ratchathewi', 'ห้วยขวาง', 'Huai Khwang',
            'บางกะปิ', 'Bang Kapi', 'มีนบุรี', 'Min Buri', 'หนองจอก', 'Nong Chok',
            'ตลิ่งชัน', 'Taling Chan', 'บางแค', 'Bang Khae', 'จตุจักร', 'Chatuchak'
        ]
        title_lower = str(title).lower()
        for loc in locations:
            if loc.lower() in title_lower:
                return loc
        return np.nan

    def load_and_merge(self):
        """Load 3 CSVs and merge into a unified schema."""
        # 1. DDProperty
        dd = pd.read_csv(os.path.join(self.raw_dir, 'ddproperty.csv'))
        dd_clean = pd.DataFrame({
            'id': dd['id'],
            'title': dd['title'],
            'description': dd['description'],
            'price_raw': dd['price'],
            'area_raw': dd['area_sqm'],
            'area_unit': 'sqm',
            'bedrooms': dd['bedrooms'],
            'bathrooms': dd['bathrooms'],
            'property_type': dd['property_type'],
            'location': dd['location'],
            'floor': dd['floor'],
            'near_bts': dd['near_bts'],
            'posted_date': dd['posted_date'],
            'source': dd['source']
        })
        
        # 2. Baania
        bn = pd.read_csv(os.path.join(self.raw_dir, 'baania.csv'))
        bn_clean = pd.DataFrame({
            'id': bn['รหัส'],
            'title': bn['หัวข้อ'],
            'description': bn['รายละเอียด'],
            'price_raw': bn['ราคา'],
            'area_raw': bn['พื้นที่_ตร.วา'],
            'area_unit': 'sqw',
            'bedrooms': bn['ห้องนอน'],
            'bathrooms': bn['ห้องน้ำ'],
            'property_type': bn['ประเภท'],
            'location': bn['ทำเล'],
            'floor': bn['ชั้น'],
            'near_bts': bn['ใกล้รถไฟฟ้า'],
            'posted_date': bn['วันที่ลง'],
            'source': bn['แหล่ง']
        })
        
        # 3. HipFlat — extract location from title since no location column
        hf = pd.read_csv(os.path.join(self.raw_dir, 'hipflat.csv'))
        hf_locations = hf['name'].apply(self._extract_location_from_title)
        
        hf_clean = pd.DataFrame({
            'id': hf['listing_id'],
            'title': hf['name'],
            'description': hf['desc'],
            'price_raw': hf['cost_usd'],
            'area_raw': hf['area'],
            'area_unit': 'sqm_split',
            'bedrooms': hf['beds'],
            'bathrooms': hf['baths'],
            'property_type': hf['type'],
            'location': hf_locations,
            'floor': hf['level'],
            'near_bts': pd.Series([np.nan]*len(hf)),
            'posted_date': hf['date'],
            'source': hf['platform'],
            'land_rai': hf['land_rai'],
            'land_ngan': hf['land_ngan'],
            'land_wah': hf['land_wah']
        })
        
        df = pd.concat([dd_clean, bn_clean, hf_clean], ignore_index=True)
        self.stats['before'] = len(df)
        print(f"  Loaded {len(dd_clean)} DDProperty + {len(bn_clean)} Baania + {len(hf_clean)} HipFlat = {len(df)} total")
        return df
        
    def deduplicate(self, df):
        """Remove duplicates."""
        initial_len = len(df)
        # Exact duplicates by id
        df = df.drop_duplicates(subset=['id'], keep='first')
        
        # Fuzzy/Cross-source duplicates: same title + location + similar price
        df = df.drop_duplicates(subset=['title', 'location', 'price_raw'], keep='first')
        
        self._log_op('deduplication', initial_len - len(df))
        return df
        
    def handle_missing(self, df):
        """Handle missing values appropriately."""
        initial_len = len(df)
        
        # Drop if both price and area missing
        mask = df['price_raw'].isna() & df['area_raw'].isna()
        df = df[~mask].copy()
        
        # Flag missing prices for imputation
        df['price_imputed'] = df['price_raw'].isna()
        
        # Drop untransformable text prices
        untransformable = ['ราคาต่อรอง', 'สอบถาม']
        df = df[~df['price_raw'].astype(str).str.strip().isin(untransformable)].copy()
        
        # Handle string bedrooms
        df['bedrooms'] = df['bedrooms'].replace({'studio': 0, '3+1': 4, 'N/A': np.nan})
        df['bedrooms'] = pd.to_numeric(df['bedrooms'], errors='coerce')
        
        # Infer bedrooms by property type
        condo_mask = (df['property_type'].str.contains('คอนโด|condo|condominium', case=False, na=False)) & df['bedrooms'].isna()
        df.loc[condo_mask, 'bedrooms'] = 1
        house_mask = (df['property_type'].str.contains('บ้าน|house|detached', case=False, na=False)) & df['bedrooms'].isna()
        df.loc[house_mask, 'bedrooms'] = 3
        townhome_mask = (df['property_type'].str.contains('ทาวน์|town', case=False, na=False)) & df['bedrooms'].isna()
        df.loc[townhome_mask, 'bedrooms'] = 2
        land_mask = (df['property_type'].str.contains('ที่ดิน|land', case=False, na=False)) & df['bedrooms'].isna()
        df.loc[land_mask, 'bedrooms'] = 0
        
        self._log_op('missing_values', initial_len - len(df))
        return df

    def filter_spam_outliers(self, df):
        """Filter out spam, outliers and fix artifacts."""
        initial_len = len(df)
        
        # === TEXT CLEANUP ===
        # HTML/Emoji artifacts in title
        df['title'] = df['title'].astype(str).str.replace(r'&amp;|<br>|\\n\\t', '', regex=True)
        df['title'] = df['title'].str.replace(r'[🏠🔥⭐\u0e00]', '', regex=True)
        
        # === SPAM REMOVAL ===
        # Phone numbers in title
        phone_mask = df['title'].str.contains(r'0\d{1,2}[-\s]?\d{3}[-\s]?\d{4}', regex=True, na=False)
        df = df[~phone_mask].copy()
        
        # Agent ads in description
        agent_keywords = r'รับฝากขาย|รับฝากเช่า|นายหน้า'
        agent_mask = df['description'].astype(str).str.contains(agent_keywords, regex=True, na=False)
        df = df[~agent_mask].copy()
        
        # Spam titles with repeated exclamation
        spam_mask = df['title'].str.contains(r'ด่วน!!!', regex=True, na=False)
        df = df[~spam_mask].copy()
        
        # === GARBAGE PROPERTY TYPES ===
        valid_types = r'คอนโด|condo|condominium|บ้าน|house|detached|ทาวน์|town|ที่ดิน|land'
        garbage_type = ~df['property_type'].astype(str).str.contains(valid_types, case=False, na=False)
        df = df[~garbage_type].copy()
        
        # === NUMERIC FIXES ===
        # Convert price_raw to numeric for checks (keep original)
        temp_price = pd.to_numeric(df['price_raw'], errors='coerce')
        
        # Remove price = 0 or price = 1 (test data)
        zero_price = (temp_price == 0) | (temp_price == 1)
        df = df[~zero_price | temp_price.isna() | df['price_raw'].apply(lambda x: isinstance(x, str) and not x.strip().isdigit())].copy()
        temp_price = pd.to_numeric(df['price_raw'], errors='coerce')
        
        # Fix negative prices/areas — take absolute
        neg_price = temp_price < 0
        df.loc[neg_price, 'price_raw'] = temp_price[neg_price].abs()
        
        temp_area = pd.to_numeric(df['area_raw'], errors='coerce')
        neg_area = temp_area < 0
        df.loc[neg_area, 'area_raw'] = temp_area[neg_area].abs()
        
        # Fix swapped price/area (price < 1000 and area > 100000)
        temp_price = pd.to_numeric(df['price_raw'], errors='coerce')
        temp_area = pd.to_numeric(df['area_raw'], errors='coerce')
        swap_mask = (temp_price < 1000) & (temp_area > 100000) & temp_price.notna() & temp_area.notna()
        if swap_mask.sum() > 0:
            df.loc[swap_mask, ['price_raw', 'area_raw']] = df.loc[swap_mask, ['area_raw', 'price_raw']].values
        
        # Remove bedrooms > 15
        df = df[(df['bedrooms'] <= 15) | df['bedrooms'].isna()].copy()
        
        # Remove area = 999999 (placeholder)
        temp_area = pd.to_numeric(df['area_raw'], errors='coerce')
        df = df[(temp_area != 999999) | temp_area.isna()].copy()
        
        # === DETECT AND FIX HipFlat USD values that are actually THB ===
        # If cost_usd > 500000 it's likely THB not USD
        hipflat_mask = df['source'] == 'HipFlat'
        temp_price_hf = pd.to_numeric(df.loc[hipflat_mask, 'price_raw'], errors='coerce')
        already_thb = temp_price_hf > 500000
        # Mark these so transformer knows not to multiply by 35
        df.loc[hipflat_mask & (temp_price_hf > 500000).reindex(df.index, fill_value=False), 'area_unit'] = 'sqm_split_thb'
        
        self._log_op('spam_outliers', initial_len - len(df))
        return df
        
    def clean(self):
        print("Starting data cleaning...")
        df = self.load_and_merge()
        print(f"\n  Phase 1: Deduplication")
        df = self.deduplicate(df)
        print(f"  Phase 2: Missing Value Handling")
        df = self.handle_missing(df)
        print(f"  Phase 3: Spam & Outlier Filtering")
        df = self.filter_spam_outliers(df)
        
        self.stats['after'] = len(df)
        os.makedirs(self.clean_dir, exist_ok=True)
        df.to_csv(os.path.join(self.clean_dir, 'cleaned_listings.csv'), index=False)
        
        print(f"\n  === Cleaning Summary ===")
        print(f"  Before: {self.stats['before']:,} records")
        print(f"  After:  {self.stats['after']:,} records")
        print(f"  Total removed: {self.stats['before'] - self.stats['after']:,}")
        for op, count in self.stats['operations'].items():
            print(f"    - {op}: {count:,}")
        
        return self.stats



# รันการทำความสะอาดข้อมูล (ลบตัวซ้ำ กรองสแปม ฯลฯ)
print("--- Step 2: Data Cleaning ---")
cleaner = DataCleaner('data/raw', 'data/cleaned')
clean_stats = cleaner.clean()

# ดูหน้าตาข้อมูลที่คลีนแล้ว
print("\nตัวอย่างข้อมูลที่ทำความสะอาดแล้ว:")
df_clean = pd.read_csv('data/cleaned/cleaned_listings.csv')
display(df_clean.head())



## 3. Data Transformation (แปลงและปรับโครงสร้าง)

In [ ]:
import os
import pandas as pd
import numpy as np
from datetime import datetime

class DataTransformer:
    """Handles feature engineering and standardization."""
    
    def __init__(self, clean_dir, trans_dir):
        self.clean_dir = clean_dir
        self.trans_dir = trans_dir
        
    def normalize_prices(self, df):
        """Parse text prices and convert USD to THB."""
        def parse_price(val, source, area_unit):
            if pd.isna(val):
                return np.nan
            
            # Numeric values
            if isinstance(val, (int, float)):
                num = float(val)
                if num <= 0:
                    return np.nan
                # HipFlat: convert USD to THB unless already marked as THB
                if source == 'HipFlat':
                    if area_unit == 'sqm_split_thb':
                        return num  # Already THB
                    return num * 35.0
                return num
                
            val_str = str(val).strip()
            
            # Thai formats: "3.5 ล้าน", "3.5ล้าน", "35 ล้านบาท"
            lan_match = re.search(r'([\d,.]+)\s*ล้าน', val_str)
            if lan_match:
                num = float(lan_match.group(1).replace(',', ''))
                return num * 1_000_000
            
            # Thai: "8 แสน"
            saen_match = re.search(r'([\d,.]+)\s*แสน', val_str)
            if saen_match:
                num = float(saen_match.group(1).replace(',', ''))
                return num * 100_000
                
            # Numeric with commas and optional "บาท": "3,500,000 บาท", "3500000"
            val_str = val_str.replace(',', '').replace('บาท', '').replace(' ', '').strip()
            try:
                num = float(val_str)
                if num <= 0:
                    return np.nan
                if source == 'HipFlat' and area_unit != 'sqm_split_thb':
                    return num * 35.0
                return num
            except:
                return np.nan
        
        import re
        df['price_thb'] = df.apply(
            lambda row: parse_price(row['price_raw'], row['source'], row.get('area_unit', '')), 
            axis=1
        )
        
        # Remove extreme prices (> 500M THB for any single listing is unrealistic for mock data)
        df.loc[df['price_thb'] > 500_000_000, 'price_thb'] = np.nan
        
        return df

    def convert_area(self, df):
        """Convert all areas to SQM."""
        def to_sqm(row):
            area = pd.to_numeric(row.get('area_raw'), errors='coerce')
            unit = row.get('area_unit', 'sqm')
            
            # For HipFlat with split rai/ngan/wah
            if unit in ['sqm_split', 'sqm_split_thb']:
                rai = pd.to_numeric(row.get('land_rai'), errors='coerce')
                ngan = pd.to_numeric(row.get('land_ngan'), errors='coerce')
                wah = pd.to_numeric(row.get('land_wah'), errors='coerce')
                
                if pd.notna(rai) and pd.notna(ngan) and pd.notna(wah):
                    # Cap rai to reasonable values (max ~10 rai for Bangkok)
                    rai = min(rai, 10)
                    ngan = min(ngan, 3)
                    sqm = (rai * 1600) + (ngan * 400) + (wah * 4)
                    if sqm > 0:
                        return sqm
                
                # Fallback to area column
                if pd.notna(area) and area > 0:
                    return area
                return np.nan
            
            if pd.isna(area) or area <= 0:
                return np.nan
                
            if unit == 'sqw':
                return area * 4.0  # ตร.วา to ตร.ม.
            
            return area  # Already in sqm
            
        df['area_sqm'] = df.apply(to_sqm, axis=1)
        
        # Cap extreme areas (max ~20,000 sqm for urban plots)
        df.loc[df['area_sqm'] > 20000, 'area_sqm'] = np.nan
        
        return df

    def categorize_location(self, df):
        """Normalize locations and assign zones."""
        loc_map = {
            'sukhumvit': 'สุขุมวิท', 'สุขุมวิท ': 'สุขุมวิท', ' สุขุมวิท': 'สุขุมวิท', 'สุขุมวิทร์': 'สุขุมวิท',
            'silom': 'สีลม', 'sathorn': 'สาทร', 'lat phrao': 'ลาดพร้าว', 'rama 9': 'พระราม9',
            'on nut': 'อ่อนนุช', 'bang na': 'บางนา', 'rangsit': 'รังสิต', 'thonburi': 'ธนบุรี',
            'phaya thai': 'พญาไท', 'ari': 'อารีย์', 'thong lo': 'ทองหล่อ', 'ekkamai': 'เอกมัย',
            'ratchathewi': 'ราชเทวี', 'huai khwang': 'ห้วยขวาง', 'bang kapi': 'บางกะปิ',
            'min buri': 'มีนบุรี', 'nong chok': 'หนองจอก', 'taling chan': 'ตลิ่งชัน',
            'bang khae': 'บางแค', 'chatuchak': 'จตุจักร'
        }
        
        def normalize_loc(loc):
            if pd.isna(loc):
                return np.nan
            loc_str = str(loc).strip().lower()
            return loc_map.get(loc_str, loc.strip() if isinstance(loc, str) else loc)
        
        df['location'] = df['location'].apply(normalize_loc)
        
        zones = {
            'CBD': ['สุขุมวิท', 'สีลม', 'สาทร', 'ทองหล่อ', 'เอกมัย'],
            'Inner City': ['ลาดพร้าว', 'พระราม9', 'อ่อนนุช', 'พญาไท', 'อารีย์', 'ราชเทวี', 'ห้วยขวาง', 'จตุจักร'],
            'Suburban': ['บางนา', 'บางกะปิ', 'ธนบุรี', 'ตลิ่งชัน', 'บางแค'],
            'Outer': ['รังสิต', 'มีนบุรี', 'หนองจอก']
        }
        
        def assign_zone(loc):
            if pd.isna(loc):
                return 'Unknown'
            for z, locs in zones.items():
                if loc in locs:
                    return z
            return 'Unknown'
            
        df['zone'] = df['location'].apply(assign_zone)
        df['district'] = df['location']
        return df

    def feature_engineering(self, df):
        """Create derived features for ML."""
        # Normalize property type
        p_map = {
            'condo': 'คอนโด', 'condominium': 'คอนโด', 
            'house': 'บ้านเดี่ยว', 'single house': 'บ้านเดี่ยว', 'detached house': 'บ้านเดี่ยว',
            'townhome': 'ทาวน์โฮม', 'townhouse': 'ทาวน์โฮม',
            'land': 'ที่ดิน', 'land plot': 'ที่ดิน'
        }
        df['property_type'] = df['property_type'].apply(
            lambda x: p_map.get(str(x).lower().strip(), x) if pd.notna(x) else x
        )
        
        # Impute missing prices using median by (property_type, zone)
        for (ptype, zone), group in df.groupby(['property_type', 'zone']):
            median_price = group['price_thb'].median()
            if pd.notna(median_price):
                mask = (df['property_type'] == ptype) & (df['zone'] == zone) & df['price_thb'].isna()
                df.loc[mask, 'price_thb'] = median_price
                df.loc[mask, 'price_imputed'] = True
        
        # Fill remaining NaN prices with overall median
        overall_median = df['price_thb'].median()
        df['price_thb'] = df['price_thb'].fillna(overall_median)
        
        # Price per sqm
        df['price_per_sqm'] = np.where(
            (df['area_sqm'] > 0) & df['area_sqm'].notna(),
            df['price_thb'] / df['area_sqm'],
            np.nan
        )
        
        # BTS
        df['has_bts'] = df['near_bts'].apply(lambda x: 1 if pd.notna(x) and str(x).strip() != '' else 0)
        
        # Age
        ref_date = datetime.now()
        df['posted_date_parsed'] = pd.to_datetime(df['posted_date'], errors='coerce', dayfirst=True)
        # Try alternative formats for unparsed dates
        unparsed = df['posted_date_parsed'].isna() & df['posted_date'].notna()
        if unparsed.any():
            df.loc[unparsed, 'posted_date_parsed'] = pd.to_datetime(
                df.loc[unparsed, 'posted_date'], errors='coerce', format='mixed'
            )
        df['age_days'] = (ref_date - df['posted_date_parsed']).dt.days.fillna(180)
        
        # Floor handling
        df['floor'] = pd.to_numeric(df['floor'], errors='coerce').fillna(0)
        
        return df

    def transform(self):
        print("Starting data transformation...")
        df = pd.read_csv(os.path.join(self.clean_dir, 'cleaned_listings.csv'), low_memory=False)
        
        print("  Step 1: Price normalization...")
        df = self.normalize_prices(df)
        print("  Step 2: Area unit conversion...")
        df = self.convert_area(df)
        print("  Step 3: Location categorization...")
        df = self.categorize_location(df)
        print("  Step 4: Feature engineering...")
        df = self.feature_engineering(df)
        
        # Drop records where both price and area are still missing
        df = df.dropna(subset=['price_thb'])
        df = df[df['area_sqm'].notna() & (df['area_sqm'] > 0)]
        
        # Remove extreme price_per_sqm outliers
        df = df[df['price_per_sqm'].notna() & (df['price_per_sqm'] > 0)]
        q99 = df['price_per_sqm'].quantile(0.99)
        df = df[df['price_per_sqm'] <= q99]
        
        cols = ['id', 'title', 'price_thb', 'area_sqm', 'price_per_sqm', 'bedrooms', 'bathrooms', 
                'property_type', 'location', 'zone', 'district', 'floor', 'near_bts', 'has_bts', 
                'posted_date', 'age_days', 'source', 'price_imputed']
        
        # Ensure cols exist
        for c in cols:
            if c not in df.columns:
                df[c] = np.nan
                
        final_df = df[cols].copy()
        
        # Fill NaN bedrooms/bathrooms
        final_df['bedrooms'] = final_df['bedrooms'].fillna(1)
        final_df['bathrooms'] = final_df['bathrooms'].fillna(1)
        
        os.makedirs(self.trans_dir, exist_ok=True)
        final_df.to_csv(os.path.join(self.trans_dir, 'final_listings.csv'), index=False)
        
        print(f"\n  === Transformation Summary ===")
        print(f"  Final records: {len(final_df):,}")
        print(f"  By zone: {final_df['zone'].value_counts().to_dict()}")
        print(f"  By type: {final_df['property_type'].value_counts().to_dict()}")
        print(f"  Price range: {final_df['price_thb'].min():,.0f} — {final_df['price_thb'].max():,.0f} THB")
        print(f"  Area range: {final_df['area_sqm'].min():,.1f} — {final_df['area_sqm'].max():,.1f} sqm")
        
        return len(final_df)



# รันการแปลงข้อมูล (Zone Mapping, Standardize ราคาและพื้นที่)
print("--- Step 3: Data Transformation ---")
transformer = DataTransformer('data/cleaned', 'data/transformed')
transformer.transform()

# ดูหน้าตาข้อมูลที่แปลงเสร็จพร้อมเข้า ML
print("\nตัวอย่างข้อมูลที่แปลงโครงสร้างแล้ว (ดูคอลัมน์ zone, price_thb, area_sqm):")
df_trans = pd.read_csv('data/transformed/final_listings.csv')
display(df_trans.head())



## 4. Machine Learning (เทรนโมเดลประเมินราคา)

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import json

class PropertyPredictor:
    """Trains models and makes predictions for property prices."""
    
    def __init__(self, data_path, model_dir):
        self.data_path = data_path
        self.model_dir = model_dir
        self.best_model = None
        self.model_columns = None
        
    def prepare_data(self):
        df = pd.read_csv(self.data_path)
        
        # Fill missing numeric values
        df['floor'] = df.apply(lambda r: 0 if r['property_type'] == 'ที่ดิน' else r['floor'], axis=1)
        df = df.dropna(subset=['price_thb', 'area_sqm']) # Essential cols
        df = df.fillna({'bedrooms': 1, 'bathrooms': 1, 'floor': 1, 'age_days': 0})
        
        # One-hot encoding
        features = df[['area_sqm', 'bedrooms', 'bathrooms', 'floor', 'zone', 'property_type', 'has_bts', 'age_days']]
        features = pd.get_dummies(features, columns=['zone', 'property_type'])
        
        X = features
        y = df['price_thb']
        self.model_columns = list(X.columns)
        
        return train_test_split(X, y, test_size=0.2, random_state=42)

    def train_and_evaluate(self):
        print("Starting model training...")
        X_train, X_test, y_train, y_test = self.prepare_data()
        
        models = {
            'Linear Regression': LinearRegression(),
            'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
            'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, random_state=42)
        }
        
        results = {}
        best_r2 = -float('inf')
        best_name = ""
        
        for name, model in models.items():
            print(f"Training {name}...")
            model.fit(X_train, y_train)
            preds = model.predict(X_test)
            
            mae = mean_absolute_error(y_test, preds)
            rmse = np.sqrt(mean_squared_error(y_test, preds))
            r2 = r2_score(y_test, preds)
            
            results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}
            
            if r2 > best_r2:
                best_r2 = r2
                self.best_model = model
                best_name = name
                
        print(f"Best model: {best_name} (R2: {best_r2:.4f})")
        
        # Save artifacts
        os.makedirs(self.model_dir, exist_ok=True)
        joblib.dump(self.best_model, os.path.join(self.model_dir, 'best_model.pkl'))
        joblib.dump(self.model_columns, os.path.join(self.model_dir, 'model_columns.pkl'))
        
        with open(os.path.join(self.model_dir, 'evaluation.json'), 'w') as f:
            json.dump(results, f, indent=4)
            
        # Feature importance if available
        if hasattr(self.best_model, 'feature_importances_'):
            fi = dict(zip(self.model_columns, self.best_model.feature_importances_))
            with open(os.path.join(self.model_dir, 'feature_importance.json'), 'w') as f:
                json.dump(fi, f, indent=4)
                
        return results

    @classmethod
    def load_model(cls, model_dir):
        inst = cls(None, model_dir)
        inst.best_model = joblib.load(os.path.join(model_dir, 'best_model.pkl'))
        inst.model_columns = joblib.load(os.path.join(model_dir, 'model_columns.pkl'))
        return inst
        
    def predict(self, area_sqm, bedrooms, bathrooms, floor, zone, property_type, has_bts):
        # Create input df
        input_data = pd.DataFrame([{
            'area_sqm': area_sqm, 'bedrooms': bedrooms, 'bathrooms': bathrooms,
            'floor': floor, 'has_bts': has_bts, 'age_days': 0,
            f'zone_{zone}': 1, f'property_type_{property_type}': 1
        }])
        
        # Reindex to match training columns
        input_data = input_data.reindex(columns=self.model_columns, fill_value=0)
        return self.best_model.predict(input_data)[0]



# รันการเทรนโมเดล Gradient Boosting
print("--- Step 4: Model Training ---")
predictor = PropertyPredictor('data/transformed/final_listings.csv', 'models')
predictor.train_and_evaluate()

